In [1]:
from IPython.core.display import HTML
with open ("../style.css", "r") as file:
    css = file.read()
HTML(css)

# Term Simplification via  Rewriting

## Type Checking

The functions in this notebook carry *type annotations*.  *Python* itself ignores these annotations, but the
type checker [*basedpyright*](https://docs.basedpyright.com) can use them to find errors before the program is
run.  In *JupyterLab*, the extension *jupyterlab-lsp* runs *basedpyright* in the background and underlines
type errors while you type.  On the command line, the command
```
basedpyright Rewrite.ipynb
```
checks the whole notebook.  The settings of the type checker are stored in the file `pyrightconfig.json` in the
directory `Python`.

Both packages, `basedpyright` and `jupyterlab-lsp`, are installed by the script `fl.sh`.  Start `jupyter lab` in the directory `Python`, so that the settings in `pyrightconfig.json` are used.

In [4]:
import string

The type `RegExp` describes regular expressions.  These are either integers, strings, or nested tuples of integers and strings.

In [5]:
type RegExp = int | str | tuple[RegExp, ...]

**Function `is_variable(s)`**
- *Input:* `s` is a regular expression.
- *Output:* `True` if `s` is a variable, i.e. a string that starts with an upper case letter, `False` otherwise.

In [ ]:
def is_variable(s: RegExp) -> bool:
    return isinstance(s, str) and s != '𝜀' and s[0] in string.ascii_uppercase

A substitution is a dictionary mapping variable names to regular expressions.

In [7]:
Subst = dict[str, RegExp]

**Function `match(pattern, term, Substitution)`**
- *Input:* `pattern` and `term` are regular expressions that may contain variables, and `Substitution` is a dictionary that maps variables to regular expressions.  The function extends `Substitution`.
- *Output:* `True` if `pattern` matches `term`, i.e. if applying `Substitution` to `pattern` yields `term`, `False` otherwise.

In [ ]:
def match(pattern: RegExp, term: RegExp, Substitution: Subst) -> bool:
    if is_variable(pattern):
        V = pattern
        if V in Substitution:
            return match(Substitution[V], term, Substitution) # type: ignore
        else:
            Substitution[V] = term                            # type: ignore
            return True
    if isinstance(pattern, str) or isinstance(pattern, int):
        return pattern == term
    if isinstance(term, str) or isinstance(term, int):
        return False
    if len(pattern) != len(term):
        return False
    if pattern[1] != term[1]:
        return False
    n = len(pattern)
    for i in range(n):
        if not match(pattern[i], term[i], Substitution):
            return False
    return True

**Function `apply(term, Substitution)`**
- *Input:* `term` is a regular expression that may contain variables and `Substitution` maps variables to regular expressions.
- *Output:* The regular expression that results from replacing the variables in `term` according to `Substitution`.

In [ ]:
def apply(term: RegExp, Substitution: Subst) -> RegExp:
    if is_variable(term):
        V = term
        if V in Substitution:
            return Substitution[V] # type: ignore
        else:
            return V
    if isinstance(term, str) or isinstance(term, int):
        return term
    return tuple(apply(arg, Substitution) for arg in term)

**Function `rewrite(term, rule)`**
- *Input:* `term` is a regular expression and `rule` is a pair `(lhs, rhs)` of regular expressions.
- *Output:* A pair `(flag, result)`: if `lhs` matches `term`, `flag` is `True` and `result` is the rewritten term, otherwise `flag` is `False` and `result` is `term`.

In [ ]:
def rewrite(term: RegExp, rule: tuple[RegExp, RegExp]) -> tuple[bool, RegExp]:
    lhs, rhs = rule
    Substitution: Subst = {}
    if match(lhs, term, Substitution):
        return True, apply(rhs, Substitution)
    else:
        return False, term

**Function `simplify_once(term, Rules)`**
- *Input:* `term` is a regular expression and `Rules` is a set of rewrite rules.
- *Output:* The term that results from applying one rule to `term` or, if no rule applies at the top, to its arguments.

In [ ]:
def simplify_once(term: RegExp, Rules: set[tuple[RegExp, RegExp]]) -> RegExp:
    if isinstance(term, str) or isinstance(term, int):
        return term
    for rule in Rules:
        flag, simple = rewrite(term, rule)
        if flag:
            return simple
    return tuple(simplify_once(arg, Rules) for arg in term)

**Function `get_rules()`**
- *Input:* No arguments.
- *Output:* The set of rewrite rules that are used to simplify regular expressions.

In [ ]:
def get_rules() -> set[tuple[RegExp, RegExp]]: 
    return { (('R', '+', 0), 'R'),
             ((0, '+', 'R'), 'R'),
             (('R', '+', 'R'), 'R'),
             (('𝜀', '+', ('R', '*')), ('R', '*')),
             ((('R', '*'), '+', '𝜀'), ('R', '*')),
             (('𝜀', '+', ('R', '⋅', ('R', '*'))), ('R', '*')),
             (('𝜀', '+', (('R', '*'), '⋅', 'R')), ('R', '*')),
             ((('R', '⋅', ('R', '*')), '+', '𝜀'), ('R', '*')),
             (((('R', '*'), '⋅', 'R'), '+', '𝜀'), ('R', '*')),
             (('S', '+', ('S', '⋅', 'T')), ('S', '⋅', ('𝜀', '+', 'T'))),
             (('S', '+', ('T', '⋅', 'S')), (('𝜀', '+', 'T'), '⋅', 'S')),
             ((0, '⋅', 'R'), 0),
             (('R', '⋅', 0), 0),
             (('𝜀', '⋅', 'R'), 'R'),
             (('R', '⋅', '𝜀'), 'R'),
             ((('𝜀', '+', 'R'), '⋅', ('R', '*')), ('R', '*')),
             ((('R', '+', '𝜀'), '⋅', ('R', '*')), ('R', '*')),
             ((('R', '*'), '⋅', ('R', '+', '𝜀')), ('R', '*')),
             ((('R', '*'), '⋅', ('𝜀', '+', 'R')), ('R', '*')),
             ((0, '*'), '𝜀'),
             (('𝜀', '*'), '𝜀'),
             ((('𝜀', '+', 'R'), '*'), ('R', '*')),
             ((('R', '+', '𝜀'), '*'), ('R', '*')),
             (('R', '+', ('S', '+', 'T')), (('R', '+', 'S'), '+', 'T')),
             (('R', '⋅', ('S', '⋅', 'T')), (('R', '⋅', 'S'), '⋅', 'T')),
             ((('R', '⋅', ('S', '*')), '⋅', ('𝜀', '+', 'S')), ('R', '⋅', ('S', '*')))
           }

**Function `simplify(t)`**
- *Input:* `t` is a regular expression.
- *Output:* The regular expression that results from applying the rewrite rules until nothing changes any more.

In [ ]:
def simplify(t: RegExp) -> RegExp:
    while True:
        old_t = t
        t     = simplify_once(t, get_rules())
        if t == old_t:
            return t